In [1]:
import numpy as np
import pandas as pd
import os
import glob 
import shutil
import time
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
import opendatasets as od
dataset_url = 'https://www.kaggle.com/competitions/signal-pose-prediction/data?select=train.csv'
od.download(dataset_url)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:

  mandojin


Your Kaggle Key:

  ········


100%|██████████████████████████████████████████████████████████████████████████████| 1.08G/1.08G [00:55<00:00, 20.9MB/s]



Extracting archive ./signal-pose-prediction/signal-pose-prediction.zip to ./signal-pose-prediction


In [5]:
annotation_df = pd.read_csv("/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/train.csv")
annotation_df.head()

,id,class
0,80a8dc1d-b216-469c-91b1-fc257562a883,0
1,3ab012da-927a-40d6-95f8-5a2f96dad360,0
2,eef92f84-5127-4063-a4a2-fee4035b1ac7,0
3,8d562739-6502-4a6c-8b83-e029b795f685,0
4,404c2e92-1101-4e62-83bf-a6fdb7dfcb9a,0


In [6]:
annotation_df.shape ,\
annotation_df['class'].value_counts()

((647, 2),
 class
 6    121
 4    116
 3    111
 5    107
 1     99
 0     50
 2     43
 Name: count, dtype: int64)

In [7]:
train_dir = '/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/train/train/'
annotation_df['id'] = annotation_df['id'].apply(lambda x : train_dir + x+ '.npy')

In [8]:
grouped_dict = {key: group['id'].to_list() for key, group in annotation_df.groupby('class')}

In [9]:
grouped_dict[0][:5]

['/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/train/train/80a8dc1d-b216-469c-91b1-fc257562a883.npy',
 '/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/train/train/3ab012da-927a-40d6-95f8-5a2f96dad360.npy',
 '/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/train/train/eef92f84-5127-4063-a4a2-fee4035b1ac7.npy',
 '/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/train/train/8d562739-6502-4a6c-8b83-e029b795f685.npy',
 '/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/train/train/404c2e92-1101-4e62-83bf-a6fdb7dfcb9a.npy']

In [10]:
def range_time(IQ_data):
    n_rd_history = 256
    frame = []
    frames = []

    for iqini in IQ_data:
        if len(frame)<n_rd_history:
            frame.append(iqini)
        else:  
            frames.append(np.copy(frame))
            frame.append(iqini)
            frame = frame[1::]
            
    return np.stack(frames)

def range_frequency(datas):
    Range_frequency_frame = []
    for data in datas:
        # Range-Doppler
        rd = np.fft.fft(data, axis=0)
        rd = np.fft.fftshift(rd, axes=0)
        rd = np.abs(rd)
        DBrd = 20 * np.log10(rd+1e-10)
        Range_frequency_frame.append(DBrd)
    return np.stack(Range_frequency_frame)

def srf_transform(complex_img, half=False):
  img = range_time(complex_img)
  img = range_frequency(img)
  if half:
    img = img[:, :img.shape[1]//2, :]
  srf_img = img.reshape(img.shape[0]*img.shape[1], img.shape[2]).real
  return srf_img.T

In [8]:
# !rm -rf /home/natthakit/sigma/preprocess_dataset

In [11]:
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/train
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/train/0
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/train/1
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/train/2
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/train/3
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/train/4
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/train/5
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/train/6

In [12]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from PIL import Image

output_dir = "/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset"  # : EDIT HERE :
dpi = 100
height = 224
width = 224

def process_image(args):
    """Processes a single image file"""
    file_path, class_id = args

    try:
        image = np.load(file_path)[:, :, 1]
        distance_img = np.abs(image)  # : EDIT HERE : select your amplitude (abs: distance, angle: velocity)
        filter_img = srf_transform(distance_img)  # : EDIT HERE : select your function

        # Save Figure
        fig, axs = plt.subplots(1, 1, figsize=(width/dpi, height/dpi))
        axs.imshow(filter_img, cmap='jet', aspect='auto')  # : EDIT HERE : select your cmap
        axs.axis('off')

        # Construct output path
        class_dir = os.path.join(output_dir, "train", str(class_id))
        os.makedirs(class_dir, exist_ok=True)  # Ensure directory exists

        filename = os.path.basename(file_path).replace('.npy', '.png')
        fig.savefig(os.path.join(class_dir, filename), dpi=dpi, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

def process_class_images(class_id, file_paths):
    """Parallel processing of a class of images"""
    args_list = [(file_path, class_id) for file_path in file_paths]
    
    with Pool(processes=cpu_count()) as pool:
        list(tqdm(pool.imap_unordered(process_image, args_list), total=len(args_list), desc=f"Processing Class {class_id}"))

if __name__ == "__main__":
    # Start multiprocessing per class
    for class_id, file_paths in grouped_dict.items():
        process_class_images(class_id, file_paths)

Processing Class 6: 100%|█████████████████████████████████████████████████████████████| 121/121 [02:01<00:00,  1.01s/it]


In [13]:
!mkdir /home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/test

In [14]:
test_file_path = glob.glob('/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/test/test'+'/*')

In [15]:
import os
import glob
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from PIL import Image

# Define dataset paths
test_file_path = glob.glob('/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/test/test/' + '/*')
output_dir = "/home/papinwit/wk2image2cap/uwb/signal-pose-prediction/preprocess_dataset/"  # : EDIT HERE :

# Image settings
dpi = 100      # : EDIT HERE :
height = 224   # : EDIT HERE :
width = 224    # : EDIT HERE :

def process_image(file_path):
    """Processes a single test image file"""
    try:
        image = np.load(file_path)[:, :, 1]
        distance_img = np.abs(image)  # : EDIT HERE : Select your amplitude (abs: distance, angle: velocity)
        filter_img = srf_transform(distance_img)  # : EDIT HERE : Select your function

        # Save Figure
        fig, axs = plt.subplots(1, 1, figsize=(width/dpi, height/dpi))
        axs.imshow(filter_img, cmap='jet', aspect='auto')  # : EDIT HERE : Select your cmap
        axs.axis('off')

        # Construct output path
        os.makedirs(os.path.join(output_dir, "test"), exist_ok=True)  # Ensure directory exists
        filename = os.path.basename(file_path).replace('.npy', '.png')
        fig.savefig(os.path.join(output_dir, "test", filename), dpi=dpi, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

if __name__ == "__main__":
    # Use multiprocessing for faster execution
    with Pool(processes=cpu_count()) as pool:
        list(tqdm(pool.imap_unordered(process_image, test_file_path), total=len(test_file_path), desc="Processing Test Images"))

Processing Test Images: 100%|█████████████████████████████████████████████████████████| 164/164 [01:47<00:00,  1.52it/s]
